# Predicting which Hogwarts house Harry Potter Characters belong to

In [83]:
import pandas as pd
import numpy as np
import scipy.stats as stats

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import make_classification
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# for clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score
)





# Download the data

In [64]:
# Load the CSV file for the training dataset
data = pd.read_csv('harry_potter_1000_students.csv')
df = pd.DataFrame(data)

df

,Blood Status,Bravery,Intelligence,Loyalty,Ambition,Dark Arts Knowledge,Quidditch Skills,Dueling Skills,Creativity,House
0,Half-blood,9,4,7,5,0,8,8,7,Gryffindor
1,Muggle-born,6,8,5,7,5,6,4,9,Ravenclaw
2,Pure-blood,1,4,7,7,1,4,4,6,Hufflepuff
3,Pure-blood,9,1,3,4,1,9,10,1,Gryffindor
4,Muggle-born,5,9,7,3,3,6,7,9,Ravenclaw
...,...,...,...,...,...,...,...,...,...,...
995,Half-blood,7,10,3,7,1,1,3,8,Ravenclaw
996,Pure-blood,7,3,2,7,8,6,7,7,Slytherin
997,Half-blood,5,10,5,3,3,5,7,10,Ravenclaw
998,Half-blood,5,6,10,4,4,6,2,4,Hufflepuff


In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Blood Status         1000 non-null   object
 1   Bravery              1000 non-null   int64 
 2   Intelligence         1000 non-null   int64 
 3   Loyalty              1000 non-null   int64 
 4   Ambition             1000 non-null   int64 
 5   Dark Arts Knowledge  1000 non-null   int64 
 6   Quidditch Skills     1000 non-null   int64 
 7   Dueling Skills       1000 non-null   int64 
 8   Creativity           1000 non-null   int64 
 9   House                1000 non-null   object
dtypes: int64(8), object(2)
memory usage: 78.3+ KB


In [66]:
# Check if there are any missing values in the DataFrame
has_missing = data.isnull().values.any()
print("Are there any missing values?", has_missing)


# Only show columns with missing values
df.isnull().sum()[df.isnull().sum() > 0]


Are there any missing values? False


Series([], dtype: int64)

In [78]:
# Check both at once
non_pureblood_slytherin = df[(df['Blood Status'].isin(['Muggle-born', 'Half-blood'])) & 
                              (df['House'] == 'Slytherin')]
print(f"Non-purebloods in Slytherin: {len(non_pureblood_slytherin)}")

Non-purebloods in Slytherin: 177


### normal dataset

In [54]:
# Prepare features and target
# Encode Blood Status (categorical feature)
le = LabelEncoder()
df['Blood Status Encoded'] = le.fit_transform(df['Blood Status'])

# Select features (all numeric columns except House)
feature_cols = ['Blood Status Encoded', 'Bravery', 'Intelligence', 'Loyalty', 
                'Ambition', 'Dark Arts Knowledge', 'Quidditch Skills', 
                'Dueling Skills', 'Creativity']

X = df[feature_cols].values
y = df['House'].values


### politically-correct-only attributes

In [67]:
# Prepare features and target
# Encode Blood Status (categorical feature)
le = LabelEncoder()
df['Blood Status Encoded'] = le.fit_transform(df['Blood Status'])

# Select features (all numeric columns except House)
feature_cols = ['Bravery', 'Intelligence', 'Loyalty', 
                'Ambition']

X = df[feature_cols].values
y = df['House'].values


In [68]:
print("\n" + "="*60)
print("HOUSE DISTRIBUTION IN ORIGINAL DATASET")
print("="*60)

# Show class distribution BEFORE split
unique_orig, counts_orig = np.unique(y, return_counts=True)
for house, count in zip(unique_orig, counts_orig):
    print(f"  {house:12s}: {count:3d} ({count/len(y)*100:.1f}%)")


HOUSE DISTRIBUTION IN ORIGINAL DATASET
  Gryffindor  : 226 (22.6%)
  Hufflepuff  : 251 (25.1%)
  Ravenclaw   : 258 (25.8%)
  Slytherin   : 265 (26.5%)


# Splitting data into train-test-split

In [72]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [73]:
X_train

array([[ 1,  7,  7,  2],
       [ 2,  1,  5,  7],
       [ 7,  7,  2,  7],
       ...,
       [ 5,  4,  6, 10],
       [ 3,  4,  7,  1],
       [ 2,  4,  8,  1]], shape=(700, 4))

# Base Classifier

In [71]:
print("\n" + "="*60)
print("HOUSE DISTRIBUTION AFTER TRAIN-TEST SPLIT")
print("="*60)

# Show class distribution
print("\nTraining set distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for house, count in zip(unique, counts):
    print(f"  {house:12s}: {count:3d} ({count/len(y_train)*100:.1f}%)")

print("\nTest set distribution:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for house, count in zip(unique_test, counts_test):
    print(f"  {house:12s}: {count:3d} ({count/len(y_test)*100:.1f}%)")

# Create and train majority class classifier
print("\n" + "="*60)
print("MAJORITY CLASS CLASSIFIER")
print("="*60)

majority_clf = DummyClassifier(strategy='most_frequent', random_state=42)
majority_clf.fit(X_train, y_train)

# Make predictions
y_pred = majority_clf.predict(X_test)

# Find the majority class
majority_house = unique[np.argmax(counts)]
print(f"\nMajority House: {majority_house}")
print(f"Strategy: Always predict '{majority_house}'")

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")



HOUSE DISTRIBUTION AFTER TRAIN-TEST SPLIT

Training set distribution:
  Gryffindor  : 158 (22.6%)
  Hufflepuff  : 176 (25.1%)
  Ravenclaw   : 181 (25.9%)
  Slytherin   : 185 (26.4%)

Test set distribution:
  Gryffindor  :  68 (22.7%)
  Hufflepuff  :  75 (25.0%)
  Ravenclaw   :  77 (25.7%)
  Slytherin   :  80 (26.7%)

MAJORITY CLASS CLASSIFIER

Majority House: Slytherin
Strategy: Always predict 'Slytherin'

Accuracy: 0.2667 (26.67%)


# Decision Tree

In [75]:
# Train Decision Tree Classifier with Hyperparameter Tuning
print("\n" + "="*60)
print("DECISION TREE WITH HYPERPARAMETER TUNING")
print("="*60)

# Define parameter distribution for Decision Tree
dt_param_dist = {
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 8, 10],
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_features': ['sqrt', 'log2', None]
}

print("\nSearching for best hyperparameters for Decision Tree...")
print("Parameter space:")
for param, values in dt_param_dist.items():
    print(f"  {param}: {values}")

# Create base Decision Tree
dt_base = DecisionTreeClassifier(random_state=42)

# Perform RandomizedSearchCV
dt_random_search = RandomizedSearchCV(
    estimator=dt_base,
    param_distributions=dt_param_dist,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

dt_random_search.fit(X_train, y_train)

print(f"\nBest parameters found:")
for param, value in dt_random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest cross-validation score: {dt_random_search.best_score_:.4f} ({dt_random_search.best_score_*100:.2f}%)")

# Use the best model
dt_clf = dt_random_search.best_estimator_

# Make predictions
y_pred_dt = dt_clf.predict(X_test)

# Evaluate Decision Tree
dt_accuracy = accuracy_score(y_test, y_pred_dt)
print(f"\nDecision Tree Test Accuracy: {dt_accuracy:.4f} ({dt_accuracy*100:.2f}%)")

print("\n" + "-"*60)
print("Classification Report:")
print("-"*60)
print(classification_report(y_test, y_pred_dt))

print("Confusion Matrix:")
print("-"*60)
houses = sorted(np.unique(y))
cm_dt = confusion_matrix(y_test, y_pred_dt, labels=houses)
print(f"\n{'':12s}", end='')
for h in houses:
    print(f"{h:12s}", end='')
print()
for i, h in enumerate(houses):
    print(f"{h:12s}", end='')
    for j in range(len(houses)):
        print(f"{cm_dt[i,j]:12d}", end='')
    print()

# Feature importance for Decision Tree
print("\n" + "-"*60)
print("Decision Tree Feature Importance:")
print("-"*60)
feature_names = feature_cols
dt_importances = dt_clf.feature_importances_
dt_indices = np.argsort(dt_importances)[::-1]

for i in range(len(feature_names)):
    print(f"{i+1}. {feature_names[dt_indices[i]]:15s}: {dt_importances[dt_indices[i]]:.4f}")



DECISION TREE WITH HYPERPARAMETER TUNING

Searching for best hyperparameters for Decision Tree...
Parameter space:
  max_depth: [3, 5, 7, 10, 15, 20, None]
  min_samples_split: [2, 5, 10, 15, 20]
  min_samples_leaf: [1, 2, 4, 8, 10]
  criterion: ['gini', 'entropy']
  splitter: ['best', 'random']
  max_features: ['sqrt', 'log2', None]
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters found:
  splitter: best
  min_samples_split: 2
  min_samples_leaf: 2
  max_features: None
  max_depth: 20
  criterion: gini

Best cross-validation score: 0.9443 (94.43%)

Decision Tree Test Accuracy: 0.9433 (94.33%)

------------------------------------------------------------
Classification Report:
------------------------------------------------------------
              precision    recall  f1-score   support

  Gryffindor       0.88      1.00      0.94        68
  Hufflepuff       0.99      0.96      0.97        75
   Ravenclaw       0.95      0.95      0.95        77
   Sly

# Random Forest

In [76]:
# Train Random Forest Classifier
print("\n" + "="*60)
print("RANDOM FOREST WITH HYPERPARAMETER TUNING")
print("="*60)

# Define parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [5, 10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'criterion': ['gini', 'entropy']
}

print("\nSearching for best hyperparameters...")
print("Parameter space:")
for param, values in param_dist.items():
    print(f"  {param}: {values}")

# Create base Random Forest
rf_base = RandomForestClassifier(random_state=42)

# Perform RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=10,  # Number of random combinations to try
    cv=3,  # 5-fold cross-validation
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,  # Use all available cores
    verbose=1
)

random_search.fit(X_train, y_train)

print(f"\nBest parameters found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest cross-validation score: {random_search.best_score_:.4f} ({random_search.best_score_*100:.2f}%)")

# Use the best model
rf_clf = random_search.best_estimator_

# Make predictions
y_pred_rf = rf_clf.predict(X_test)

# Evaluate Random Forest
rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(f"\nRandom Forest Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")


# Feature importance
print("\n" + "-"*60)
print("Feature Importance:")
print("-"*60)
feature_names = feature_cols
importances = rf_clf.feature_importances_
indices = np.argsort(importances)[::-1]

for i in range(len(feature_names)):
    print(f"{i+1}. {feature_names[indices[i]]:15s}: {importances[indices[i]]:.4f}")


RANDOM FOREST WITH HYPERPARAMETER TUNING

Searching for best hyperparameters...
Parameter space:
  n_estimators: [50, 100, 200, 300, 500]
  max_depth: [5, 10, 15, 20, 25, None]
  min_samples_split: [2, 5, 10, 15]
  min_samples_leaf: [1, 2, 4, 8]
  max_features: ['sqrt', 'log2', None]
  bootstrap: [True, False]
  criterion: ['gini', 'entropy']
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters found:
  n_estimators: 100
  min_samples_split: 5
  min_samples_leaf: 2
  max_features: None
  max_depth: 20
  criterion: entropy
  bootstrap: False

Best cross-validation score: 0.9400 (94.00%)

Random Forest Accuracy: 0.9367 (93.67%)

------------------------------------------------------------
Feature Importance:
------------------------------------------------------------
1. Ambition       : 0.3563
2. Loyalty        : 0.2811
3. Bravery        : 0.2394
4. Intelligence   : 0.1232


# Clustering

In [84]:
# K-MEANS CLUSTERING
print("\n" + "="*60)
print("K-MEANS CLUSTERING (UNSUPERVISED)")
print("="*60)

# Prepare features for clustering (only numerical attributes, no Blood Status)
cluster_features = ['Bravery', 'Intelligence', 'Loyalty', 'Ambition', 
                    'Dark Arts Knowledge', 'Quidditch Skills', 'Dueling Skills', 'Creativity']

X_cluster = df[cluster_features].values

# Standardize features (important for K-means)
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

print("\nFeatures used for clustering:")
for feat in cluster_features:
    print(f"  - {feat}")

# Apply K-means with 4 clusters (for 4 houses)
print("\nApplying K-means with 4 clusters...")
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster_scaled)

# Add cluster labels to dataframe
df['Cluster'] = clusters

# Calculate clustering metrics
silhouette = silhouette_score(X_cluster_scaled, clusters)
ari = adjusted_rand_score(df['House'], clusters)

print(f"\nSilhouette Score: {silhouette:.4f} (higher is better, range: -1 to 1)")
print(f"Adjusted Rand Index: {ari:.4f} (measures similarity to true houses, range: -1 to 1)")

# Show cluster distribution
print("\n" + "-"*60)
print("Cluster Distribution:")
print("-"*60)
unique_clusters, cluster_counts = np.unique(clusters, return_counts=True)
for cluster, count in zip(unique_clusters, cluster_counts):
    print(f"  Cluster {cluster}: {count:3d} students ({count/len(df)*100:.1f}%)")

# Cross-tabulation: Clusters vs Houses
print("\n" + "-"*60)
print("Cluster vs House Cross-tabulation:")
print("-"*60)
crosstab = pd.crosstab(df['Cluster'], df['House'], margins=True)
print(crosstab)

# Find which cluster corresponds to which house (based on majority)
print("\n" + "-"*60)
print("Cluster-to-House Mapping (based on majority):")
print("-"*60)
for cluster in sorted(df['Cluster'].unique()):
    house_counts = df[df['Cluster'] == cluster]['House'].value_counts()
    majority_house = house_counts.index[0]
    majority_count = house_counts.iloc[0]
    total = len(df[df['Cluster'] == cluster])
    print(f"  Cluster {cluster} → {majority_house:12s} ({majority_count}/{total} = {majority_count/total*100:.1f}%)")

# Show cluster centers (average traits per cluster)
print("\n" + "-"*60)
print("Cluster Centers (Average Traits):")
print("-"*60)
cluster_centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
print(f"\n{'Feature':20s}", end='')
for i in range(4):
    print(f"Cluster {i:1d}  ", end='')
print()
print("-" * 60)
for i, feat in enumerate(cluster_features):
    print(f"{feat:20s}", end='')
    for j in range(4):
        print(f"{cluster_centers_original[j,i]:10.2f}", end='')
    print()

# Interpretation
print("\n" + "-"*60)
print("Clustering Interpretation:")
print("-"*60)
print(f"K-means found natural groupings in the data with {silhouette:.2f} silhouette score.")
print(f"The Adjusted Rand Index of {ari:.2f} shows how well clusters match actual houses.")
print("(ARI=1: perfect match, ARI=0: random, ARI<0: worse than random)")


K-MEANS CLUSTERING (UNSUPERVISED)

Features used for clustering:
  - Bravery
  - Intelligence
  - Loyalty
  - Ambition
  - Dark Arts Knowledge
  - Quidditch Skills
  - Dueling Skills
  - Creativity

Applying K-means with 4 clusters...

Silhouette Score: 0.3233 (higher is better, range: -1 to 1)
Adjusted Rand Index: 0.9973 (measures similarity to true houses, range: -1 to 1)

------------------------------------------------------------
Cluster Distribution:
------------------------------------------------------------
  Cluster 0: 265 students (26.5%)
  Cluster 1: 250 students (25.0%)
  Cluster 2: 226 students (22.6%)
  Cluster 3: 259 students (25.9%)

------------------------------------------------------------
Cluster vs House Cross-tabulation:
------------------------------------------------------------
House    Gryffindor  Hufflepuff  Ravenclaw  Slytherin   All
Cluster                                                    
0                 0           0          0        265   265
1  